# Fine-tuning DistilBERT Base Model for Extractive Question Answering with Stanford Question Answering Dataset (SQuAD) 

## Libraries Installation

In [1]:
pip install -q transformers datasets evaluate protobuf==3.20.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 37.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requi

## GPU Check

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Loading the dataset and model

In [3]:
from datasets import load_dataset

squad = load_dataset("squad")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [4]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

## Preprocessing the Training Dataset

In [5]:
max_length = 384
stride = 128

def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label is (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [6]:
train_dataset = squad["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=squad["train"].column_names,
)

len(squad["train"]), len(train_dataset)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

(87599, 88729)

## Preprocessing the Validation Dataset

In [7]:
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [8]:
validation_dataset = squad["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=squad["validation"].column_names,
)
len(squad["validation"]), len(validation_dataset)

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

(10570, 10822)

## Post processing

In [ ]:
from tqdm.auto import tqdm
import collections
import numpy as np
import evaluate

# Constants for answer extraction
n_best = 20
max_answer_length = 30

# Load SQuAD metric for evaluation
metric = evaluate.load("squad")

def compute_metrics(start_logits, end_logits, features, examples):
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # Loop through all features associated with that example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Skip answers with a length that is either < 0 or > max_answer_length
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # Select the answer with the best score
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

2025-11-24 19:55:55.079416: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764014155.264088      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764014155.318721      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## Fine-tuning the model with the Trainer API

In [10]:
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint).to(device)

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-squad",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=1, # @todo increase
    weight_decay=0.01,
    fp16=True,
    report_to="none",
)

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
)

trainer.train()

Step,Training Loss
1,5.958000
100,4.550000
200,3.710600
300,3.090300
400,2.621400
500,2.498800
600,2.415800
700,2.343400
800,2.270100
900,2.282200


TrainOutput(global_step=11092, training_loss=1.5391867679247906, metrics={'train_runtime': 1794.6106, 'train_samples_per_second': 49.442, 'train_steps_per_second': 6.181, 'total_flos': 8694539184554496.0, 'train_loss': 1.5391867679247906, 'epoch': 1.0})

## Evaluation

In [13]:
predictions, _, _ = trainer.predict(validation_dataset)
start_logits, end_logits = predictions
compute_metrics(start_logits, end_logits, validation_dataset, squad["validation"])

  0%|          | 0/10570 [00:00<?, ?it/s]

{'exact_match': 73.8221381267739, 'f1': 82.73834276879933}

## Test

In [ ]:
from transformers import pipeline

# Createas pipeline for base model
base_qa_pipeline = pipeline(
    task="question-answering",
    model="distilbert-base-cased"
)

# Creates a pipeline for fine-tuned model
tuned_qa_pipeline = pipeline(
    task="question-answering",
    model=trainer.model,
    tokenizer=tokenizer
)

# Test cases
test_cases = [
    {
        "context": """
        Retrieval-Augmented Generation (RAG), introduced by Lewis et al. in 2020, is designed to provide Large Language Models (LLMs) with additional information from an external knowledge source.
        The main purpose of RAG is to generate more accurate and contextual answers while reducing hallucinations.
        In RAG, factual knowledge is separated from the LLMs reasoning capability and stored in an external knowledge source.
        This technique is analogous to taking an open-book exam: “RAG is to LLMs what an open-book exam is to humans”.
        """,
        "question": "What is the primary function of Retrieval-Augmented Generation (RAG) in LLM applications?"
    },
    {
        "context": """
        GPT-3 is an autoregressive language model released by OpenAI in 2020.
        This model is characterized by its scale, which includes 96 Transformer decoder layers.
        The model possesses 175 billion parameters and was trained using 300 billion tokens from several sources.
        Furthermore, GPT-3 has a 2048-token-long context window, which represents the amount of text the model can consider when generating a response.
        """,
        "question": "How many parameters does the GPT-3 model have?"
    }, 
    {
        "context": """
        The transformer architecture was introduced by Vaswani et al. in the paper “Attention Is All You Need” (2017).
        The architecture eliminates Recurrent Neural Networks (RNNs) and uses only attention mechanisms.
        While the quality of MT was improved, the architecture especially drastically improves the training speed because it deals with sequential data in parallel, allowing the use of GPUs/TPUs.
        Language Models like Gemini and ChatGPT are based on transformers.
        """,
        "question": "Which research paper introduced the Transformer architecture and in what year?"
    },
    {
        "context": """
        Long short-term memory (LSTM) models were developed to address long-term dependencies, solving issues found in traditional Recurrent Neural Networks (RNNs) that often exhibit short-term memory.
        LSTMs introduce the cell state, which is key to their functionality.
        The cell state acts as the long-term memory of the LSTM, designed to carry information across many time steps while undergoing minimal modifications.
        In contrast, the hidden state represents the short-term, immediate output, used for predictions and communication with other layers.
        """,
        "question": "What role does the cell state play within the Long short-term memory (LSTM) architecture?"
    },
    {
        "context": """
        Word embeddings are designed not only to reflect similarity but also to preserve semantic and syntactic relationships.
        These relationships are observed in the difference vectors between words.
        This property allows for analogies or queries to be expressed as equations
        Furthermore, these sophisticated relationships can be leveraged in other applications, such as assisting with machine translation.
        """,
        "question": "How do word embeddings encode semantic and syntactic relationships between words?"
    }
]

# Run comparison
for i, test_case in enumerate(test_cases, 1):
    context = test_case["context"].strip()
    question = test_case["question"]

    # Get prediction from base model
    base_result = base_qa_pipeline(question=question, context=context)
    base_answer = base_result["answer"]
    base_score = base_result["score"]

    # Get prediction from fine-tuned model
    tuned_result = tuned_qa_pipeline(question=question, context=context)
    tuned_answer = tuned_result["answer"]
    tuned_score = tuned_result["score"]

    print(f"Test case #{i}\n")
    print(f"Context: {context}\n")
    print(f"Question: {question}\n")

    print(f"Base model:\n")
    print(f"- Answer: {base_answer}\n")
    print(f"- Confidence: {base_score:.4f}\n")

    print(f"Fine-tuned model:\n")
    print(f"- Answer: {tuned_answer}\n")
    print(f"- Confidence: {tuned_score:.4f}\n")